# Cross-City Investment Intelligence — Madrid · Barcelona · Málaga
**Inputs:** `../Data/interim/{city}_listings_clean.parquet` for each city  
**Prerequisite:** run each city's `01_cleaning.ipynb` first.

Compares the three markets on the metrics that drive Airbnb investment returns:
nightly rate, annual revenue, occupancy, competitive density, and regulatory risk.
Use this notebook to identify which city and which neighbourhood offer the strongest
risk-adjusted yield for a short-term rental investment.

In [ ]:
import sys
sys.path.insert(0, '..')

import pathlib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations

CITY_COLORS = {'Madrid': '#E63946', 'Barcelona': '#457B9D', 'Málaga': '#2A9D8F'}
CITY_ORDER  = ['Madrid', 'Barcelona', 'Málaga']

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 60)

In [ ]:
_PATHS = {
    'Madrid':    '../Data/interim/madrid_listings_clean.parquet',
    'Barcelona': '../Data/interim/barcelona_listings_clean.parquet',
    'Málaga':    '../Data/interim/malaga_listings_clean.parquet',
}

frames = {}
for city, path in _PATHS.items():
    p = pathlib.Path(path)
    if not p.exists():
        print(f'[WARNING] {city}: file not found — {path}')
        continue
    df = pd.read_parquet(path)
    if df.empty:
        print(f'[WARNING] {city}: parquet is empty — re-run 01_cleaning.ipynb first')
        continue
    df['city'] = city
    frames[city] = df
    print(f'{city:12s}  {df.shape[0]:>6,} listings | {df.shape[1]} cols')

if not frames:
    raise RuntimeError('No city data loaded — check parquet paths above.')

loaded_cities = [c for c in CITY_ORDER if c in frames]
all_df = pd.concat(frames.values(), join='outer', ignore_index=True)
all_df['city'] = pd.Categorical(all_df['city'], categories=loaded_cities, ordered=True)

print(f'\nCombined: {all_df.shape[0]:,} listings × {all_df.shape[1]} cols')
print(f'Cities  : {loaded_cities}')

## 1 · Market size & composition

How large is each market? Listing volume sets the competitive baseline: a larger supply base means more competition for the same tourist demand. Room and property type mix reveal the structural character of each market — whether it is dominated by entire apartments (higher revenue potential, higher regulatory scrutiny) or private rooms.

In [ ]:
counts = all_df.groupby('city', observed=True).size().reset_index(name='listings')

fig = px.bar(
    counts, x='city', y='listings', color='city',
    color_discrete_map=CITY_COLORS, text='listings',
    title='Total cleaned listings per city',
    labels={'listings': 'Listing count', 'city': ''},
    category_orders={'city': CITY_ORDER},
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, yaxis_title='Listings')
fig.show()
display(counts.set_index('city'))

In [ ]:
room_mix = (
    all_df.groupby(['city', 'room_type'], observed=True)
    .size().reset_index(name='count')
)
room_mix['pct'] = room_mix.groupby('city')['count'].transform(lambda x: x / x.sum() * 100)

fig = px.bar(
    room_mix, x='city', y='pct', color='room_type', barmode='stack',
    title='Room type mix by city (%)',
    labels={'pct': 'Share (%)', 'city': '', 'room_type': 'Room type'},
    category_orders={'city': CITY_ORDER},
    text=room_mix['pct'].round(1).astype(str) + '%',
)
fig.update_traces(textposition='inside')
fig.update_layout(yaxis_title='Share (%)')
fig.show()

room_pivot = room_mix.pivot(index='room_type', columns='city', values='pct').round(1).fillna(0)
display(room_pivot.sort_values(loaded_cities[0], ascending=False))

In [ ]:
prop_mix = (
    all_df.groupby(['city', 'property_type_std'], observed=True)
    .size().reset_index(name='count')
)
prop_mix['pct'] = prop_mix.groupby('city')['count'].transform(lambda x: x / x.sum() * 100)

fig = px.bar(
    prop_mix, x='city', y='pct', color='property_type_std', barmode='stack',
    title='Property type mix by city (%)',
    labels={'pct': 'Share (%)', 'city': '', 'property_type_std': 'Property type'},
    category_orders={'city': CITY_ORDER},
)
fig.update_layout(yaxis_title='Share (%)')
fig.show()

prop_pivot = prop_mix.pivot(index='property_type_std', columns='city', values='pct').round(1).fillna(0)
display(prop_pivot.sort_values(loaded_cities[0], ascending=False))

## 2 · Nightly rate comparison

Nightly price is the primary lever for annual revenue. Higher rates drive more income per booked night but may reduce demand and occupancy. We compare the absolute price level and the structure across room types to understand where the pricing premium lies.

In [ ]:
fig = px.box(
    all_df, x='city', y='price', color='city',
    color_discrete_map=CITY_COLORS,
    title='Nightly price distribution by city',
    labels={'price': 'Price per night (€)', 'city': ''},
    category_orders={'city': CITY_ORDER},
    points=False,
)
fig.update_layout(showlegend=False)
fig.show()

price_stats = (
    all_df.groupby('city', observed=True)['price']
    .agg(['count', 'median', 'mean', 'std'])
    .round(1)
)
display(price_stats)

groups = [g['price'].dropna().values for _, g in all_df.groupby('city', observed=True)]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    sig = 'significant' if p < 0.05 else 'not significant'
    print(f'\nKruskal-Wallis: H = {stat:.2f},  p = {p:.2e}  ({sig} price difference across cities)')

In [ ]:
price_rt = (
    all_df[all_df['room_type'].isin(['Entire home/apt', 'Private room'])]
    .groupby(['city', 'room_type'], observed=True)['price']
    .median()
    .reset_index(name='median_price')
)

fig = px.bar(
    price_rt, x='room_type', y='median_price', color='city',
    barmode='group', color_discrete_map=CITY_COLORS,
    title='Median nightly price: entire home vs. private room',
    labels={'median_price': 'Median price (€)', 'room_type': '', 'city': 'City'},
    category_orders={'city': CITY_ORDER},
    text=price_rt['median_price'].apply(lambda x: f'€{x:.0f}'),
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_title='Median price (€)')
fig.show()

display(price_rt.pivot(index='room_type', columns='city', values='median_price').round(1))

## 3 · Revenue & occupancy

Annual revenue (`estimated_revenue_l365d`) and occupancy (`estimated_occupancy_l365d`) are the core investment metrics. Revenue reflects total income potential; occupancy reflects demand strength. The ideal market has **both** high rates and high occupancy — a city with elevated prices but low occupancy may underperform one with moderate prices and strong consistent demand.

> **Note:** `estimated_revenue_l365d` may not be available for all cities depending on the InsideAirbnb data export. Coverage is reported below.

In [ ]:
rev_avail = (
    all_df.groupby('city', observed=True)['estimated_revenue_l365d']
    .apply(lambda x: x.notna().sum())
    .rename('non_null')
)
total_city = all_df.groupby('city', observed=True).size().rename('total')
rev_cov = pd.concat([rev_avail, total_city], axis=1)
rev_cov['coverage_%'] = (rev_cov['non_null'] / rev_cov['total'] * 100).round(1)
print('Revenue data coverage per city:')
display(rev_cov)

cities_with_revenue = rev_cov[rev_cov['non_null'] > 0].index.tolist()
if not cities_with_revenue:
    print('\n[NOTE] No city has revenue estimates — occupancy is used as the primary investment metric.')
else:
    print(f'\nRevenue available for: {cities_with_revenue}')

In [ ]:
if cities_with_revenue:
    rev_df = all_df[
        all_df['city'].isin(cities_with_revenue) &
        all_df['estimated_revenue_l365d'].notna()
    ].copy()

    rev_stats = (
        rev_df.groupby('city', observed=True)['estimated_revenue_l365d']
        .agg(['count', 'median', 'mean', 'std'])
        .round(0)
    )
    display(rev_stats)

    fig = px.box(
        rev_df, x='city', y='estimated_revenue_l365d', color='city',
        color_discrete_map=CITY_COLORS,
        title='Estimated annual revenue distribution by city',
        labels={'estimated_revenue_l365d': 'Estimated annual revenue (€)', 'city': ''},
        category_orders={'city': CITY_ORDER},
        points=False,
    )
    fig.update_layout(showlegend=False)
    fig.show()

    if len(cities_with_revenue) >= 2:
        print('Pairwise Mann-Whitney U tests (revenue):')
        for c1, c2 in combinations(cities_with_revenue, 2):
            g1 = rev_df[rev_df['city'] == c1]['estimated_revenue_l365d'].dropna()
            g2 = rev_df[rev_df['city'] == c2]['estimated_revenue_l365d'].dropna()
            _, p = mannwhitneyu(g1, g2, alternative='two-sided')
            print(f'  {c1} vs {c2}: p = {p:.2e}')
else:
    print('Revenue data not available — see occupancy section below.')

In [ ]:
all_df['occupancy_rate'] = all_df['estimated_occupancy_l365d'] / 365

fig = px.box(
    all_df, x='city', y='estimated_occupancy_l365d', color='city',
    color_discrete_map=CITY_COLORS,
    title='Estimated occupancy distribution by city (days booked per year)',
    labels={'estimated_occupancy_l365d': 'Occupancy (days/year)', 'city': ''},
    category_orders={'city': CITY_ORDER},
    points=False,
)
fig.update_layout(showlegend=False)
fig.show()

occ_stats = (
    all_df.groupby('city', observed=True)['estimated_occupancy_l365d']
    .agg(['count', 'median', 'mean', 'std'])
    .round(1)
)
occ_stats['median_rate_%'] = (occ_stats['median'] / 365 * 100).round(1)
display(occ_stats)

In [ ]:
city_agg = all_df.groupby('city', observed=True).agg(
    median_price=('price', 'median'),
    median_occupancy=('estimated_occupancy_l365d', 'median'),
    n=('price', 'count'),
).reset_index()

if cities_with_revenue:
    city_rev = (
        all_df[all_df['estimated_revenue_l365d'].notna()]
        .groupby('city', observed=True)['estimated_revenue_l365d']
        .median().reset_index(name='median_revenue')
    )
    city_agg = city_agg.merge(city_rev, on='city', how='left')
    size_col = 'median_revenue'
else:
    size_col = 'n'

fig = px.scatter(
    city_agg, x='median_price', y='median_occupancy',
    color='city', size=size_col, text='city',
    color_discrete_map=CITY_COLORS,
    title='City-level quadrant: nightly price vs. occupancy',
    labels={
        'median_price': 'Median nightly price (€)',
        'median_occupancy': 'Median occupancy (days/year)',
    },
)
fig.update_traces(textposition='top center')
fig.add_vline(x=city_agg['median_price'].mean(), line_dash='dash', annotation_text='Price avg')
fig.add_hline(y=city_agg['median_occupancy'].mean(), line_dash='dash', annotation_text='Occupancy avg')
fig.update_layout(showlegend=False, height=500)
fig.show()
display(city_agg.set_index('city').drop(columns=['n']).round(1))

## 4 · Top-performing districts

District selection is as important as city selection. Within each city, revenue and occupancy can vary by a factor of 2–3× across neighbourhoods. The charts below identify the top performers using the best available metric — annual revenue where data exists, otherwise occupancy — then aggregate results into a cross-city investment atlas.

In [ ]:
primary_metric = 'estimated_revenue_l365d' if cities_with_revenue else 'estimated_occupancy_l365d'
primary_label  = 'Median annual revenue (€)' if cities_with_revenue else 'Median occupancy (days/year)'

for city in frames.keys():
    city_df = all_df[all_df['city'] == city].copy()
    if primary_metric == 'estimated_revenue_l365d':
        city_df = city_df[city_df[primary_metric].notna()]
    if city_df.empty:
        print(f'{city}: no {primary_metric} data — skipping.')
        continue

    # Adaptive minimum: 1 % of the city's filtered listings, floored at 5
    min_n = max(5, len(city_df) // 100)

    district_stats = (
        city_df.groupby('neighbourhood_group_cleansed')
        .agg(
            metric_val=(primary_metric, 'median'),
            median_price=('price', 'median'),
            median_occupancy=('estimated_occupancy_l365d', 'median'),
            n=('price', 'count'),
        )
        .query(f'n >= {min_n}')
        .sort_values('metric_val', ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={'metric_val': primary_label})
    )

    if 'revenue' in primary_label.lower():
        fmt = lambda v: f'€{v:,.0f}'
    else:
        fmt = lambda v: f'{v:.0f} d'

    fig = px.bar(
        district_stats,
        x=primary_label, y='neighbourhood_group_cleansed',
        orientation='h',
        title=f'{city} — Top districts by {primary_label.lower()}',
        labels={'neighbourhood_group_cleansed': 'District'},
        text=district_stats[primary_label].apply(fmt),
        color=primary_label,
        color_continuous_scale='Blues',
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
    fig.show()
    display(district_stats.set_index('neighbourhood_group_cleansed').round(1))
    print()

In [ ]:
atlas = pd.DataFrame()
atlas_rows = []
for city in frames.keys():
    city_df = all_df[all_df['city'] == city].copy()
    if primary_metric == 'estimated_revenue_l365d':
        city_df = city_df[city_df[primary_metric].notna()]
    if city_df.empty:
        continue

    # Same adaptive minimum as the chart above
    min_n = max(5, len(city_df) // 100)

    top = (
        city_df.groupby('neighbourhood_group_cleansed')
        .agg(
            metric_val=(primary_metric, 'median'),
            median_price=('price', 'median'),
            median_occupancy=('estimated_occupancy_l365d', 'median'),
            n=('price', 'count'),
        )
        .query(f'n >= {min_n}')
        .sort_values('metric_val', ascending=False)
        .head(5)
        .reset_index()
    )
    top['city'] = city
    atlas_rows.append(top)

if atlas_rows:
    atlas = (
        pd.concat(atlas_rows, ignore_index=True)
        .rename(columns={
            'neighbourhood_group_cleansed': 'District',
            'metric_val': primary_label,
            'median_price': 'Median price (€)',
            'median_occupancy': 'Median occupancy (days)',
            'n': 'Listings',
        })
        [['city', 'District', 'Median price (€)', 'Median occupancy (days)', primary_label, 'Listings']]
        .sort_values(primary_label, ascending=False)
        .reset_index(drop=True)
    )
    print(f'Investment Atlas — top 5 districts per city, ranked by {primary_label.lower()}:')
    display(atlas)
else:
    print('No district data available.')

## 5 · Competitive landscape

Understanding the host mix helps assess market entry difficulty. Markets dominated by professional operators (6+ listings) are harder to compete in — these hosts optimise pricing, availability, and guest experience at scale. The superhost rate signals the quality threshold new entrants must meet. Review scores show how high that bar already is.

In [ ]:
SCALE_ORDER = ['Peer-to-Peer (1)', 'Small portfolio (2–5)', 'Commercial (6+)']

def host_scale(x):
    if x == 1:
        return 'Peer-to-Peer (1)'
    if x <= 5:
        return 'Small portfolio (2–5)'
    return 'Commercial (6+)'

all_df['host_scale'] = all_df['calculated_host_listings_count'].apply(host_scale)

scale_mix = (
    all_df.groupby(['city', 'host_scale'], observed=True)
    .size().reset_index(name='count')
)
scale_mix['pct'] = scale_mix.groupby('city')['count'].transform(lambda x: x / x.sum() * 100)

fig = px.bar(
    scale_mix, x='city', y='pct', color='host_scale', barmode='stack',
    title='Host portfolio scale distribution by city (%)',
    labels={'pct': 'Share (%)', 'city': '', 'host_scale': 'Host type'},
    category_orders={'city': CITY_ORDER, 'host_scale': SCALE_ORDER},
    color_discrete_sequence=['#2ecc71', '#f39c12', '#e74c3c'],
    text=scale_mix['pct'].round(1).astype(str) + '%',
)
fig.update_traces(textposition='inside')
fig.update_layout(yaxis_title='Share (%)')
fig.show()

scale_pivot = scale_mix.pivot(index='host_scale', columns='city', values='pct').round(1)
display(scale_pivot.reindex(SCALE_ORDER))

In [ ]:
sh_rate = (
    all_df.groupby('city', observed=True)['host_is_superhost']
    .mean().mul(100).round(1).reset_index(name='superhost_pct')
)
fig1 = px.bar(
    sh_rate, x='city', y='superhost_pct', color='city',
    color_discrete_map=CITY_COLORS,
    title='Superhost rate by city (%)',
    labels={'superhost_pct': 'Superhost rate (%)', 'city': ''},
    category_orders={'city': CITY_ORDER},
    text=sh_rate['superhost_pct'].apply(lambda x: f'{x:.1f}%'),
)
fig1.update_traces(textposition='outside')
fig1.update_layout(showlegend=False)
fig1.show()

fig2 = px.box(
    all_df.dropna(subset=['review_scores_rating']),
    x='city', y='review_scores_rating', color='city',
    color_discrete_map=CITY_COLORS,
    title='Review score distribution by city',
    labels={'review_scores_rating': 'Review score (1–5)', 'city': ''},
    category_orders={'city': CITY_ORDER},
    points=False,
)
fig2.update_layout(showlegend=False)
fig2.show()

rating_stats = (
    all_df.groupby('city', observed=True)['review_scores_rating']
    .agg(['count', 'median', 'mean', 'std']).round(3)
)
display(rating_stats)

## 6 · Regulatory environment

Spain requires short-term rental licenses in all three regions covered here: **VT** (*Vivienda con Fines Turísticos*) in Madrid and Andalucía, and **HUT** (*Habitatge d'Ús Turístic*) in Catalonia. License status is self-reported by hosts in InsideAirbnb data. A high share of **Unknown / missing** licenses signals regulatory exposure and potential enforcement risk — a key factor when assessing the long-term viability of an STR investment.

In [ ]:
def classify_license(lic):
    if pd.isna(lic) or str(lic).strip().lower() in ('', 'unknown'):
        return 'Unknown / missing'
    low = str(lic).lower()
    if 'exempt' in low or 'exento' in low or 'exenta' in low:
        return 'Exempt'
    if 'proceso' in low or 'process' in low:
        return 'In process'
    return 'Licensed'

all_df['license_status'] = all_df['license'].apply(classify_license)
all_df['is_compliant'] = all_df['license_status'].isin(['Licensed', 'Exempt'])

LIC_ORDER = ['Licensed', 'Exempt', 'In process', 'Unknown / missing']
LIC_COLORS = {
    'Licensed': '#2ecc71', 'Exempt': '#f39c12',
    'In process': '#3498db', 'Unknown / missing': '#e74c3c',
}

lic_mix = (
    all_df.groupby(['city', 'license_status'], observed=True)
    .size().reset_index(name='count')
)
lic_mix['pct'] = lic_mix.groupby('city')['count'].transform(lambda x: x / x.sum() * 100)

fig = px.bar(
    lic_mix, x='city', y='pct', color='license_status', barmode='stack',
    title='License status by city (%)',
    labels={'pct': 'Share (%)', 'city': '', 'license_status': 'License status'},
    category_orders={'city': CITY_ORDER, 'license_status': LIC_ORDER},
    color_discrete_map=LIC_COLORS,
    text=lic_mix['pct'].round(1).astype(str) + '%',
)
fig.update_traces(textposition='inside')
fig.update_layout(yaxis_title='Share (%)')
fig.show()

compliance = (
    all_df.groupby('city', observed=True)['is_compliant']
    .mean().mul(100).round(1).rename('compliance_rate_%')
)
print('Effective compliance rate (Licensed + Exempt) per city:')
display(compliance)

## 7 · City investment scorecard

A single-view summary of all key metrics in investment-decision order: revenue potential first, then demand signals, then competitive and regulatory risk factors. The radar chart normalises each metric to [0, 1] so the overall investment profile of each city can be compared at a glance.

In [ ]:
scorecard = all_df.groupby('city', observed=True).agg(
    total_listings=('id', 'count'),
    median_price_eur=('price', 'median'),
    median_occupancy_days=('estimated_occupancy_l365d', 'median'),
    occupancy_rate_pct=('occupancy_rate', lambda x: x.median() * 100),
    mean_review_score=('review_scores_rating', 'mean'),
    superhost_pct=('host_is_superhost', lambda x: x.mean() * 100),
    commercial_pct=('host_scale', lambda x: (x == 'Commercial (6+)').mean() * 100),
    compliance_pct=('is_compliant', lambda x: x.mean() * 100),
).round(1)

if cities_with_revenue:
    rev_med = (
        all_df[all_df['estimated_revenue_l365d'].notna()]
        .groupby('city', observed=True)['estimated_revenue_l365d']
        .median().round(0)
    )
    scorecard.insert(2, 'median_annual_revenue_eur', rev_med)

display(scorecard.T.rename_axis('Metric'))

In [ ]:
radar_cols = {
    'Nightly price': 'median_price_eur',
    'Occupancy (%)': 'occupancy_rate_pct',
    'Review score': 'mean_review_score',
    'Superhost rate': 'superhost_pct',
    'Compliance rate': 'compliance_pct',
}
available_radar = {k: v for k, v in radar_cols.items() if v in scorecard.columns}
radar_data = scorecard[list(available_radar.values())].copy()

if len(radar_data) > 1:
    radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)
else:
    radar_norm = radar_data.apply(lambda col: col / col.max() if col.max() > 0 else col)

categories = list(available_radar.keys())
fig = go.Figure()
for city in radar_norm.index:
    vals = radar_norm.loc[city].tolist()
    vals.append(vals[0])
    fig.add_trace(go.Scatterpolar(
        r=vals,
        theta=categories + [categories[0]],
        fill='toself',
        name=str(city),
        line_color=CITY_COLORS.get(str(city)),
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='City investment profile (normalised — higher is better on all axes)',
    showlegend=True,
    height=550,
)
fig.show()

In [ ]:
print('=' * 65)
print('INVESTMENT SUMMARY')
print('=' * 65)

top_price     = scorecard['median_price_eur'].idxmax()
low_price     = scorecard['median_price_eur'].idxmin()
top_price_val = scorecard.at[top_price, 'median_price_eur']
low_price_val = scorecard.at[low_price, 'median_price_eur']
print(f'\nNightly rate   highest: {top_price} (€{top_price_val:.0f}/night median)')
print(f'               lowest:  {low_price} (€{low_price_val:.0f}/night median)')

if 'median_annual_revenue_eur' in scorecard.columns and scorecard['median_annual_revenue_eur'].notna().any():
    top_rev     = scorecard['median_annual_revenue_eur'].idxmax()
    top_rev_val = scorecard.at[top_rev, 'median_annual_revenue_eur']
    print(f'\nAnnual revenue highest: {top_rev} (€{top_rev_val:,.0f}/year median)')

top_occ     = scorecard['median_occupancy_days'].idxmax()
top_occ_val = scorecard.at[top_occ, 'median_occupancy_days']
print(f'\nOccupancy      highest: {top_occ} ({top_occ_val:.0f} days/year median)')

most_comm      = scorecard['commercial_pct'].idxmax()
least_comm     = scorecard['commercial_pct'].idxmin()
most_comm_val  = scorecard.at[most_comm, 'commercial_pct']
least_comm_val = scorecard.at[least_comm, 'commercial_pct']
print(f'\nCompetition    most commercial:  {most_comm} ({most_comm_val:.0f}% commercial operators)')
print(f'               least commercial: {least_comm} ({least_comm_val:.0f}%)')

top_comp     = scorecard['compliance_pct'].idxmax()
low_comp     = scorecard['compliance_pct'].idxmin()
top_comp_val = scorecard.at[top_comp, 'compliance_pct']
low_comp_val = scorecard.at[low_comp, 'compliance_pct']
print(f'\nRegulatory     most compliant: {top_comp} ({top_comp_val:.0f}%)')
print(f'               most risky:     {low_comp} ({low_comp_val:.0f}% compliant)')

if not atlas.empty:
    best             = atlas.iloc[0]
    best_district    = best['District']
    best_city        = best['city']
    best_metric_val  = best[primary_label]
    print(f'\nTop district   {best_district}, {best_city}')
    print(f'               {primary_label}: {best_metric_val:,.0f}')

print('\n' + '=' * 65)
print('Next step: link Idealista purchase prices to compute true yield')
print('(yield = annual revenue / purchase price). See finance module.')

## Key findings

_Run the notebook to populate the investment summary above, then record narrative observations here._

**Interpreting the scorecard:**
- **High revenue + high occupancy** — strong demand-supply balance; the best investment case
- **High commercial share (6+)** — professional competition at scale; hard to differentiate without active management
- **Low compliance rate** — regulatory enforcement risk; factor legal exposure into return projections
- **High superhost rate** — elevated quality bar; budget for professional management or hands-on hosting

**Next step:** link Idealista property price data with `estimated_revenue_l365d` per neighbourhood to compute actual annual yield (`revenue / purchase_price`). This bridges the Airbnb performance layer with the core buy/sell decision.